# Unsupervised Feature Engineering + Preprocessing

This notebook is the **unsupervised counterpart** to the supervised
`Data_Preprocessing.ipynb` workflow.

The overall preprocessing philosophy is kept the same:

1. Detect identifier-like columns
2. Create missing-value indicator flags
3. Handle categorical missing values
4. Handle numerical missing values
5. Handle optional temporal columns
6. Handle skewed numerical features
7. Group rare categorical labels
8. Encode categorical features
9. MinMax scaling

### What changes for unsupervised learning?

There is **no target variable**, so target-guided encoding and target-based
feature selection cannot be used.

Instead, categorical features use **frequency encoding**:

> Each category is replaced by the proportion of rows containing that category.

This keeps each categorical feature as **one feature**, avoiding the feature
explosion that one-hot encoding can cause.

There is also **no Lasso / target-based feature selection** in this notebook.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

%matplotlib inline

from scipy import stats

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

pd.pandas.set_option("display.max_columns", None)


## Config

Adjust these for your dataset.


In [2]:
# ==========================================================
# CONFIG
# ==========================================================

INPUT_FILE = "train.csv"

# Default 80/20 split
TEST_SIZE = 0.20
RANDOM_STATE = 42

RARE_LABEL_THRESHOLD = 0.01
SKEW_THRESHOLD = 0.75

# Optional temporal transformation
TEMPORAL_COLS = []
TEMPORAL_REFERENCE_COL = None

# Create outputs automatically in the same working folder.
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

X_TRAIN_OUTPUT = OUTPUT_DIR / f"X_train_{RUN_TIMESTAMP}.csv"
X_TEST_OUTPUT = OUTPUT_DIR / f"X_test_{RUN_TIMESTAMP}.csv"

print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Run timestamp: {RUN_TIMESTAMP}")


Output folder: C:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs
Run timestamp: 20260811_194153


## Load dataset

Because there is no target variable, the complete uploaded dataset is treated
as the unsupervised feature matrix.


In [3]:
# Load the complete dataset first.
full_dataset = pd.read_csv(INPUT_FILE)

print(
    f"Loaded {full_dataset.shape[0]} rows, "
    f"{full_dataset.shape[1]} columns"
)

# Split BEFORE fitting any preprocessing step.
train_dataset, test_dataset = train_test_split(
    full_dataset,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)

print(f"Training rows: {len(train_dataset)}")
print(f"Test rows: {len(test_dataset)}")
print(f"Test size: {TEST_SIZE}")

# All preprocessing below is fitted using the training split only.
dataset = train_dataset.copy()
test = test_dataset.copy()

dataset.head()


Loaded 1460 rows, 81 columns
Training rows: 1168
Test rows: 292
Test size: 0.2


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1957,1957,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,Gd,CBlock,TA,TA,No,Rec,922,Unf,0,392,1314,GasA,TA,Y,SBrkr,1314,0,0,1314,1,0,1,0,3,1,TA,5,Typ,0,NaN,Attchd,1957.0,RFn,1,294,TA,TA,Y,250,0,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal,145000
1,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,7,1993,1994,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,Gd,TA,PConc,Gd,TA,No,Unf,0,Unf,0,799,799,GasA,Gd,Y,SBrkr,799,772,0,1571,0,0,2,1,3,1,TA,7,Typ,1,TA,Attchd,1993.0,RFn,2,380,TA,TA,Y,0,40,0,0,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal,178000
2,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Feedr,Norm,1Fam,1Story,5,7,1910,1950,Gable,CompShg,MetalSd,Wd Sdng,NaN,0.0,TA,TA,CBlock,Fa,TA,No,Unf,0,Unf,0,796,796,GasA,Gd,Y,FuseA,796,0,0,796,0,0,1,0,2,1,TA,4,Typ,0,NaN,NaN,NaN,NaN,0,0,NaN,NaN,P,328,0,164,0,0,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,85000
3,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,SWISU,Feedr,Norm,1Fam,1.5Fin,5,7,1937,1950,Gable,CompShg,Wd Sdng,Wd Sdng,BrkFace,252.0,TA,TA,BrkTil,Gd,TA,No,ALQ,569,Unf,0,162,731,GasA,Ex,Y,SBrkr,981,787,0,1768,1,0,1,1,3,1,Gd,7,Typ,2,TA,Detchd,1939.0,Unf,1,240,TA,TA,Y,0,0,264,0,0,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal,175000
4,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Norm,Norm,1Fam,1.5Fin,5,6,1924,1950,Gable,CompShg,BrkFace,Wd Sdng,NaN,0.0,TA,TA,BrkTil,TA,TA,No,LwQ,218,Unf,0,808,1026,GasA,TA,Y,SBrkr,1026,665,0,1691,0,0,2,0,3,1,Gd,6,Typ,1,Gd,Detchd,1924.0,Unf,1,308,TA,TA,Y,0,0,242,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,127000


## Drop identifier-like columns (universal — no hardcoded column name)

Any column where every value is unique is treated as an identifier-like column.
These columns generally identify rows rather than describe them and are therefore
removed from the feature matrix.

The ID is preserved separately so it can be reattached to the final output.


In [4]:
id_like_cols = [
    col
    for col in dataset.columns
    if dataset[col].nunique(dropna=False) == len(dataset)
]

print(
    f"Identifier-like columns detected: {id_like_cols}"
)

ID_COLS = id_like_cols.copy()

if ID_COLS:

    dataset_ids = dataset[ID_COLS].copy()
    test_ids = test[ID_COLS].copy()

    dataset = dataset.drop(
        columns=ID_COLS
    )

    test = test.drop(
        columns=[
            col
            for col in ID_COLS
            if col in test.columns
        ]
    )

else:

    dataset_ids = pd.DataFrame(index=dataset.index)
    test_ids = pd.DataFrame(index=test.index)

print(
    f"Feature matrix after ID removal: "
    f"{dataset.shape}"
)


Identifier-like columns detected: ['Id']
Feature matrix after ID removal: (1168, 80)


## Add missing-value indicator columns

For every column with missing values, add a companion `<col>_nan` flag
(1 if missing, 0 otherwise) before filling.

This preserves the information that a value was originally missing.


In [5]:
features_with_nan = [
    f
    for f in dataset.columns
    if dataset[f].isnull().sum() > 0
]

for feature in features_with_nan:

    dataset[feature + "_nan"] = np.where(
        dataset[feature].isnull(),
        1,
        0
    )

print(
    f"Added {len(features_with_nan)} "
    "missing-value indicator columns"
)

features_with_nan


Added 19 missing-value indicator columns


['LotFrontage',
 'Alley',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

## Handle missing values — categorical features

Categorical missing values are replaced with `"Missing"`.

This is the same missing-value treatment used in the supervised workflow.


In [6]:
features_nan_cat = [
    f
    for f in dataset.columns
    if (
        dataset[f].isnull().sum() > 0
        and dataset[f].dtype == "O"
    )
]

for feature in features_nan_cat:

    pct = np.round(
        dataset[feature].isnull().mean(),
        4
    )

    print(
        f"{feature} - {pct} missing"
    )

for feature in features_nan_cat:

    dataset[feature] = (
        dataset[feature]
        .fillna("Missing")
    )

print(
    "\nRemaining categorical missing values:",
    dataset[features_nan_cat].isnull().sum().sum()
)


Alley - 0.9366 missing
MasVnrType - 0.5848 missing
BsmtQual - 0.024 missing
BsmtCond - 0.024 missing
BsmtExposure - 0.024 missing
BsmtFinType1 - 0.024 missing
BsmtFinType2 - 0.024 missing
Electrical - 0.0009 missing
FireplaceQu - 0.4683 missing
GarageType - 0.0548 missing
GarageFinish - 0.0548 missing
GarageQual - 0.0548 missing
GarageCond - 0.0548 missing
PoolQC - 0.9949 missing
Fence - 0.8005 missing
MiscFeature - 0.9606 missing

Remaining categorical missing values: 0


## Handle missing values — numerical features

Numerical missing values are filled using the **training-data median**.

The medians are stored so the same values can later be applied to new data
without re-fitting the preprocessing logic.


In [7]:
features_nan_num = [
    f
    for f in dataset.columns
    if (
        dataset[f].isnull().sum() > 0
        and dataset[f].dtype != "O"
    )
]

for feature in features_nan_num:

    pct = np.round(
        dataset[feature].isnull().mean(),
        4
    )

    print(
        f"{feature} - {pct} missing"
    )

train_medians = {
    feature: dataset[feature].median()
    for feature in features_nan_num
}

for feature, median_value in train_medians.items():

    dataset[feature] = (
        dataset[feature]
        .fillna(median_value)
    )

print(
    "\nRemaining numerical missing values:",
    dataset[features_nan_num].isnull().sum().sum()
)


LotFrontage - 0.1858 missing
MasVnrArea - 0.0051 missing
GarageYrBlt - 0.0548 missing

Remaining numerical missing values: 0


## Handle temporal columns (optional, dataset-specific)

Only runs if `TEMPORAL_COLS` and `TEMPORAL_REFERENCE_COL` are set in the config.

For example, a `YearBuilt` column can be converted into an age-like feature using
a reference year.


In [8]:
if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:

    for feature in TEMPORAL_COLS:

        dataset[feature] = (
            dataset[TEMPORAL_REFERENCE_COL]
            - dataset[feature]
        )

    print(
        f"Converted temporal columns: {TEMPORAL_COLS}"
    )

else:

    print(
        "No temporal columns configured — skipping."
    )


No temporal columns configured — skipping.


## Handle skewed numeric features (auto-detected, not hardcoded)

Any numeric feature, excluding missing-value indicator columns, with
`abs(skew) > SKEW_THRESHOLD` is considered skewed.

Only strictly positive features are log-transformed because `np.log()` requires
positive values.


In [9]:
nan_flag_cols = [
    f + "_nan"
    for f in features_with_nan
]

numeric_cols = (
    dataset
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

candidate_cols = [
    c
    for c in numeric_cols
    if c not in nan_flag_cols
]

skewed = (
    dataset[candidate_cols]
    .apply(
        lambda x: stats.skew(
            x.dropna()
        )
    )
)

skewed_features = (
    skewed[
        abs(skewed) > SKEW_THRESHOLD
    ]
    .index
    .tolist()
)

# Only log-transform strictly positive columns.
skewed_features = [
    c
    for c in skewed_features
    if (dataset[c] > 0).all()
]

for feature in skewed_features:

    dataset[feature] = np.log(
        dataset[feature]
    )

print(
    f"Log-transformed {len(skewed_features)} "
    f"skewed columns: {skewed_features}"
)


Log-transformed 6 skewed columns: ['MSSubClass', 'LotFrontage', 'LotArea', '1stFlrSF', 'GrLivArea', 'SalePrice']


## Handle rare categorical labels

Categories making up less than `RARE_LABEL_THRESHOLD` of rows are grouped into
`"Rare_var"`.

This keeps the same rare-category handling philosophy as the supervised notebook.
The kept-category list is stored per feature so it can be reused for new data.


In [10]:
categorical_features = [
    f
    for f in dataset.columns
    if dataset[f].dtype == "O"
]

frequent_labels = {}

for feature in categorical_features:

    freq = (
        dataset[feature]
        .value_counts()
        / len(dataset)
    )

    kept = freq[
        freq > RARE_LABEL_THRESHOLD
    ].index

    frequent_labels[feature] = kept

    dataset[feature] = np.where(
        dataset[feature].isin(kept),
        dataset[feature],
        "Rare_var"
    )

print(
    f"Applied rare-label grouping to "
    f"{len(categorical_features)} categorical columns"
)


Applied rare-label grouping to 43 categorical columns


## Frequency encoding

### Why this replaces target-guided encoding

The supervised workflow ranks categories using the mean target value.
That is impossible here because there is **no target**.

Instead, each category is replaced by its frequency in the dataset.

Example:

```text
City
Delhi      → 0.42
Mumbai     → 0.31
Kolkata    → 0.18
Chennai    → 0.09
```

The important property is that **one categorical feature remains one numerical
feature**. We therefore avoid the 81 → 250+ feature expansion that can occur
with one-hot encoding.


In [11]:
frequency_mappings = {}

for feature in categorical_features:

    frequencies = (
        dataset[feature]
        .value_counts(
            normalize=True
        )
    )

    frequency_mappings[feature] = (
        frequencies.to_dict()
    )

    dataset[feature] = (
        dataset[feature]
        .map(
            frequency_mappings[feature]
        )
    )

print(
    f"Frequency-encoded "
    f"{len(categorical_features)} categorical columns"
)

dataset.head()


Frequency-encoded 43 categorical columns


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,2.995732,0.791096,4.248495,9.035987,0.996575,0.936644,0.624144,0.906678,0.999144,0.703767,0.94863,0.154966,0.859589,0.990582,0.837329,0.494007,5,6,1957,1957,0.775685,0.983733,0.148116,0.141267,0.584760,0.0,0.622432,0.104452,0.431507,0.446062,0.894692,0.65839,0.089041,922,0.86387,0,392,1314,0.976027,0.297089,0.928938,0.916952,7.180831,0,0,7.180831,1,0,1,0,3,1,0.504281,5,0.928082,0,0.468322,0.593322,1957.0,0.290240,1,294,0.898973,0.908390,0.916096,250,0,0,0,0,0,0.994863,0.800514,0.960616,0,6,2010,0.866438,0.825342,11.884489,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1
1,4.094345,0.791096,4.077537,8.966611,0.996575,0.936644,0.337329,0.906678,0.999144,0.703767,0.94863,0.055651,0.859589,0.990582,0.837329,0.308219,6,7,1993,1994,0.775685,0.983733,0.359589,0.351027,0.584760,0.0,0.332192,0.871575,0.445205,0.422089,0.894692,0.65839,0.295377,0,0.86387,0,799,799,0.976027,0.166952,0.928938,0.916952,6.683361,772,0,7.359468,0,0,2,1,3,1,0.504281,7,0.928082,1,0.215753,0.593322,1993.0,0.290240,2,380,0.898973,0.908390,0.916096,0,40,0,0,0,0,0.994863,0.800514,0.960616,0,5,2009,0.866438,0.825342,12.089539,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
2,3.401197,0.791096,4.204693,9.079890,0.996575,0.936644,0.624144,0.906678,0.999144,0.703767,0.94863,0.074486,0.056507,0.990582,0.837329,0.494007,5,7,1910,1950,0.775685,0.983733,0.148116,0.141267,0.584760,0.0,0.622432,0.871575,0.431507,0.024829,0.894692,0.65839,0.295377,0,0.86387,0,796,796,0.976027,0.166952,0.928938,0.059075,6.679599,0,0,6.679599,0,0,1,0,2,1,0.504281,4,0.928082,0,0.468322,0.054795,1980.0,0.054795,0,0,0.054795,0.054795,0.021404,328,0,164,0,0,0,0.994863,0.109589,0.960616,0,5,2008,0.866438,0.825342,11.350407,0,1,1,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,1
3,3.912023,0.791096,4.094345,8.881836,0.996575,0.936644,0.624144,0.906678,0.999144,0.189212,0.94863,0.017979,0.056507,0.990582,0.837329,0.103596,5,7,1937,1950,0.775685,0.983733,0.146404,0.141267,0.313356,252.0,0.622432,0.871575,0.099315,0.422089,0.894692,0.65839,0.152397,569,0.86387,0,162,731,0.976027,0.499144,0.928938,0.916952,6.888572,787,0,7.477604,1,0,1,1,3,1,0.402397,7,0.928082,2,0.215753,0.263699,1939.0,0.410959,1,240,0.898973,0.908390,0.916096,0,0,264,0,0,0,0.994863,0.109589,0.960616,0,6,2007,0.866438,0.825342,12.072541,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
4,3.912023,0.791096,3.912023,8.517193,0.996575,0.025685,0.624144,0.906678,0.999144,0.703767,0.94863,0.017979,0.859589,0.990582,0.837329,0.103596,5,6,1924,1950,0.775685,0.983733,0.034247,0.141267,0.584760,0.0,0.622432,0.871575,0.099315,0.446062,0.894692,0.65839,0.053082,218,0.86387,0,808,1026,0.976027,0.297089,0.928938,0.916952,6.933423,665,0,7.433075,0,0,2,0,3,1,0.402397,6,0.928082,1,0.261130,0.263699,1924.0,0.410959,1,308,0.898973,0.908390,0.916096,0,0,242,0,0,0,0.994863,0.800514,0.960616,0,5,2010,0.866438,0.825342,11.751942,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1


## Numeric validation

At this point all categorical columns should have been converted to numerical
frequency values.

This check makes sure the final matrix contains only numeric features before
scaling.


In [12]:
non_numeric_cols = [
    col
    for col in dataset.columns
    if not pd.api.types.is_numeric_dtype(
        dataset[col]
    )
]

print(
    "Non-numeric columns remaining:",
    non_numeric_cols
)

if non_numeric_cols:

    raise ValueError(
        "Non-numeric features remain after "
        "frequency encoding."
    )

print(
    "All features are numeric."
)


Non-numeric columns remaining: []
All features are numeric.


## Feature scaling

MinMaxScaler is fitted on the complete unsupervised feature matrix.

Unlike the supervised workflow, there is no target column to exclude and no
target-based feature selection after scaling.


In [13]:
scalable_features = dataset.columns.tolist()

scaler = MinMaxScaler()

scaler.fit(
    dataset[scalable_features]
)

dataset_scaled = pd.DataFrame(
    scaler.transform(
        dataset[scalable_features]
    ),
    columns=scalable_features,
    index=dataset.index
)

dataset_scaled.head()


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,0.000000,1.0,0.445638,0.365182,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.0,1.0,1.000000,0.444444,0.625,0.615942,0.116667,1.0,1.0,0.403382,0.379747,1.000000,0.000000,1.000000,0.116371,0.968750,1.000000,1.0,1.0,0.239748,0.163359,1.0,0.0,0.167808,0.215057,1.0,0.594502,1.0,1.000000,0.518336,0.000000,0.0,0.484528,0.333333,0.0,0.333333,0.0,0.375,0.333333,1.000000,0.250000,1.0,0.000000,1.000000,1.000000,0.518182,0.661058,0.25,0.207334,1.000000,1.000000,1.0,0.291715,0.000000,0.000000,0.0,0.0,0.0,1.0,1.00000,1.0,0.0,0.454545,1.00,1.0,1.0,0.465304,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,0.487992,1.0,0.382360,0.351604,1.0,1.0,0.535368,1.0,1.0,1.000000,1.0,0.309524,1.000000,1.0,1.0,0.615929,0.555556,0.750,0.876812,0.733333,1.0,1.0,1.000000,1.000000,1.000000,0.000000,0.526536,1.000000,1.000000,0.943205,1.0,1.0,1.000000,0.000000,1.0,0.0,0.342038,0.130769,1.0,0.333333,1.0,1.000000,0.330077,0.373850,0.0,0.547721,0.000000,0.0,0.666667,0.5,0.375,0.333333,1.000000,0.416667,1.0,0.333333,0.444444,1.000000,0.845455,0.661058,0.50,0.267983,1.000000,1.000000,1.0,0.000000,0.073126,0.000000,0.0,0.0,0.0,1.0,1.00000,1.0,0.0,0.363636,0.75,1.0,1.0,0.532294,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,0.180103,1.0,0.429425,0.373775,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,0.440476,0.051567,1.0,1.0,1.000000,0.444444,0.750,0.275362,0.000000,1.0,1.0,0.403382,0.379747,1.000000,0.000000,1.000000,1.000000,0.968750,0.002028,1.0,1.0,1.000000,0.000000,1.0,0.0,0.340753,0.130278,1.0,0.333333,1.0,0.060918,0.328654,0.000000,0.0,0.307217,0.000000,0.0,0.333333,0.0,0.250,0.333333,1.000000,0.166667,1.0,0.000000,1.000000,0.075000,0.727273,0.000000,0.00,0.000000,0.056459,0.045933,0.0,0.382730,0.000000,0.297101,0.0,0.0,0.0,1.0,0.12851,1.0,0.0,0.363636,0.50,1.0,1.0,0.290818,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
3,0.407007,1.0,0.388581,0.335012,1.0,1.0,1.000000,1.0,1.0,0.266178,1.0,0.047619,0.051567,1.0,1.0,0.192920,0.444444,0.750,0.471014,0.000000,1.0,1.0,0.398551,0.379747,0.526866,0.182874,1.000000,1.000000,0.210938,0.943205,1.0,1.0,0.473186,0.100815,1.0,0.0,0.069349,0.119640,1.0,1.000000,1.0,1.000000,0.407736,0.381114,0.0,0.589512,0.333333,0.0,0.333333,0.5,0.375,0.333333,0.786355,0.416667,1.0,0.666667,0.444444,0.433824,0.354545,1.000000,0.25,0.169252,1.000000,1.000000,1.0,0.000000,0.000000,0.478261,0.0,0.0,0.0,1.0,0.12851,1.0,0.0,0.454545,0.25,1.0,1.0,0.526741,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.407007,1.0,0.321097,0.263645,1.0,0.0,1.000000,1.0,1.0,1.000000,1.0,0.047619,1.000000,1.0,1.0,0.192920,0.444444,0.625,0.376812,0.000000,1.0,1.0,0.082126,0.379747,1.000000,0.000000,1.000000,1.000000,0.210938,1.000000,1.0,1.0,0.107256,0.038625,1.0,0.0,0.345890,0.167921,1.0,0.594502,1.0,1.000000,0.424709,0.322034,0

## Save engineered unsupervised dataset

This is the processed feature matrix **before any optional dimensionality
reduction**.

No Lasso or target-based feature selection is applied because there is no target.


In [14]:
# Save the processed training split.
data = dataset_scaled.copy()

if ID_COLS:

    for col in reversed(ID_COLS):

        data.insert(
            0,
            col,
            dataset_ids[col].reset_index(drop=True)
        )

data.to_csv(
    X_TRAIN_OUTPUT,
    index=False
)

print(
    f"Saved X_train: {X_TRAIN_OUTPUT}"
)

print(
    f"X_train shape: {data.shape}"
)

data.head()


Saved X_train: c:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs\X_train_20260811_194153.csv
X_train shape: (1168, 100)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,255,0.000000,1.0,0.445638,0.365182,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.0,1.0,1.000000,0.444444,0.625,0.615942,0.116667,1.0,1.0,0.403382,0.379747,1.000000,0.000000,1.000000,0.116371,0.968750,1.000000,1.0,1.0,0.239748,0.163359,1.0,0.0,0.167808,0.215057,1.0,0.594502,1.0,1.000000,0.518336,0.000000,0.0,0.484528,0.333333,0.0,0.333333,0.0,0.375,0.333333,1.000000,0.250000,1.0,0.000000,1.000000,1.000000,0.518182,0.661058,0.25,0.207334,1.000000,1.000000,1.0,0.291715,0.000000,0.000000,0.0,0.0,0.0,1.0,1.00000,1.0,0.0,0.454545,1.00,1.0,1.0,0.465304,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,1067,0.487992,1.0,0.382360,0.351604,1.0,1.0,0.535368,1.0,1.0,1.000000,1.0,0.309524,1.000000,1.0,1.0,0.615929,0.555556,0.750,0.876812,0.733333,1.0,1.0,1.000000,1.000000,1.000000,0.000000,0.526536,1.000000,1.000000,0.943205,1.0,1.0,1.000000,0.000000,1.0,0.0,0.342038,0.130769,1.0,0.333333,1.0,1.000000,0.330077,0.373850,0.0,0.547721,0.000000,0.0,0.666667,0.5,0.375,0.333333,1.000000,0.416667,1.0,0.333333,0.444444,1.000000,0.845455,0.661058,0.50,0.267983,1.000000,1.000000,1.0,0.000000,0.073126,0.000000,0.0,0.0,0.0,1.0,1.00000,1.0,0.0,0.363636,0.75,1.0,1.0,0.532294,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,639,0.180103,1.0,0.429425,0.373775,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,0.440476,0.051567,1.0,1.0,1.000000,0.444444,0.750,0.275362,0.000000,1.0,1.0,0.403382,0.379747,1.000000,0.000000,1.000000,1.000000,0.968750,0.002028,1.0,1.0,1.000000,0.000000,1.0,0.0,0.340753,0.130278,1.0,0.333333,1.0,0.060918,0.328654,0.000000,0.0,0.307217,0.000000,0.0,0.333333,0.0,0.250,0.333333,1.000000,0.166667,1.0,0.000000,1.000000,0.075000,0.727273,0.000000,0.00,0.000000,0.056459,0.045933,0.0,0.382730,0.000000,0.297101,0.0,0.0,0.0,1.0,0.12851,1.0,0.0,0.363636,0.50,1.0,1.0,0.290818,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
3,800,0.407007,1.0,0.388581,0.335012,1.0,1.0,1.000000,1.0,1.0,0.266178,1.0,0.047619,0.051567,1.0,1.0,0.192920,0.444444,0.750,0.471014,0.000000,1.0,1.0,0.398551,0.379747,0.526866,0.182874,1.000000,1.000000,0.210938,0.943205,1.0,1.0,0.473186,0.100815,1.0,0.0,0.069349,0.119640,1.0,1.000000,1.0,1.000000,0.407736,0.381114,0.0,0.589512,0.333333,0.0,0.333333,0.5,0.375,0.333333,0.786355,0.416667,1.0,0.666667,0.444444,0.433824,0.354545,1.000000,0.25,0.169252,1.000000,1.000000,1.0,0.000000,0.000000,0.478261,0.0,0.0,0.0,1.0,0.12851,1.0,0.0,0.454545,0.25,1.0,1.0,0.526741,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,381,0.407007,1.0,0.321097,0.263645,1.0,0.0,1.000000,1.0,1.0,1.000000,1.0,0.047619,1.000000,1.0,1.0,0.192920,0.444444,0.625,0.376812,0.000000,1.0,1.0,0.082126,0.379747,1.000000,0.000000,1.000000,1.000000,0.210938,1.000000,1.0,1.0,0.107256,0.038625,1.0,0.0,0.345890,0.167921,1.0,0.594502,1.0,1.00

## Feature-count sanity check

This cell explicitly checks whether preprocessing has unexpectedly increased the
feature count.

Frequency encoding should keep each categorical feature as one feature. Missing
indicators are the only intentional source of additional columns in this workflow.
Identifier columns are removed from the feature matrix and optionally preserved
separately in the output.


In [15]:
original_columns = full_dataset.shape[1]

original_feature_columns = (
    original_columns
    - len(ID_COLS)
)

added_missing_indicators = len(
    features_with_nan
)

final_feature_columns = len(
    dataset_scaled.columns
)

print(
    f"Original dataset columns: "
    f"{original_columns}"
)

print(
    f"Identifier columns removed: "
    f"{len(ID_COLS)}"
)

print(
    f"Original feature columns after ID removal: "
    f"{original_feature_columns}"
)

print(
    f"Missing indicators added: "
    f"{added_missing_indicators}"
)

print(
    f"Final processed training feature columns: "
    f"{final_feature_columns}"
)

expected_upper_bound = (
    original_feature_columns
    + added_missing_indicators
)

if final_feature_columns <= expected_upper_bound:

    print(
        "PASS: No categorical one-hot feature explosion."
    )

else:

    print(
        "WARNING: Feature count increased beyond the "
        "expected missing-indicator expansion."
    )


Original dataset columns: 81
Identifier columns removed: 1
Original feature columns after ID removal: 80
Missing indicators added: 19
Final processed training feature columns: 99
PASS: No categorical one-hot feature explosion.


## Final output preview

The final matrix is ready for downstream unsupervised algorithms such as
clustering, dimensionality reduction, or anomaly detection.


In [16]:
# Apply the identical fitted unsupervised pipeline to X_test.
# No preprocessing parameter is fitted using X_test.

test_features = test.copy()

# Same missing-value indicator flags
for feature in features_with_nan:

    if feature in test_features.columns:

        test_features[feature + "_nan"] = np.where(
            test_features[feature].isnull(),
            1,
            0
        )

# Same categorical missing-value handling
for feature in features_nan_cat:

    if feature in test_features.columns:

        test_features[feature] = (
            test_features[feature]
            .fillna("Missing")
        )

# Same numerical missing-value handling using TRAIN medians
for feature, median_value in train_medians.items():

    if feature in test_features.columns:

        test_features[feature] = (
            test_features[feature]
            .fillna(median_value)
        )

# Same temporal transformation
if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:

    for feature in TEMPORAL_COLS:

        if feature in test_features.columns:

            test_features[feature] = (
                test_features[TEMPORAL_REFERENCE_COL]
                - test_features[feature]
            )

# Same skew transformations learned from TRAIN
for feature in skewed_features:

    if feature in test_features.columns:

        test_features[feature] = np.log(
            test_features[feature]
        )

# Same rare-label grouping learned from TRAIN
for feature in categorical_features:

    if feature in test_features.columns:

        kept = frequent_labels[feature]

        test_features[feature] = np.where(
            test_features[feature].isin(kept),
            test_features[feature],
            "Rare_var"
        )

# Same frequency encoding learned from TRAIN.
# Unseen categories receive frequency 0.
for feature in categorical_features:

    if feature in test_features.columns:

        test_features[feature] = (
            test_features[feature]
            .map(frequency_mappings[feature])
            .fillna(0.0)
        )

# Ensure the same feature order as training.
test_features = test_features[
    scalable_features
]

# Same scaler: transform only.
test_scaled = pd.DataFrame(
    scaler.transform(
        test_features
    ),
    columns=scalable_features,
    index=test_features.index
)

test_final = test_scaled.copy()

# Preserve ID columns if detected.
if ID_COLS:

    for col in reversed(ID_COLS):

        test_final.insert(
            0,
            col,
            test_ids[col].reset_index(drop=True)
        )

test_final.to_csv(
    X_TEST_OUTPUT,
    index=False
)

print(
    f"Saved X_test: {X_TEST_OUTPUT}"
)

print(
    f"X_test shape: {test_final.shape}"
)

print(
    f"X_train shape: {data.shape}"
)

print(
    f"X_test shape: {test_final.shape}"
)

test_final.head(20)


Saved X_test: c:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs\X_test_20260811_194153.csv
X_test shape: (292, 100)
X_train shape: (1168, 100)
X_test shape: (292, 100)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,893,0.000000,1.000000,0.445638,0.365508,1.0,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,0.267857,1.000000,1.0,1.000000,1.000000,0.555556,0.875,0.659420,0.883333,0.236633,1.0,0.410628,0.374684,1.000000,0.000000,1.000000,1.000000,0.968750,1.000000,1.0,1.000000,0.946372,0.117470,1.0,0.000000,0.169521,0.173322,1.0,0.594502,1.0,1.000000,0.439892,0.000000,0.0,0.411200,0.000000,0.5,0.333333,0.0,0.375,0.333333,1.000000,0.333333,1.0,0.000000,1.000000,1.000000,0.572727,0.661058,0.25,0.186178,1.000000,1.000000,1.0,0.224037,0.000000,0.000000,0.0,0.000000,0.0,1.0,0.128510,1.000000,0.000000,0.090909,0.00,1.0,1.000000,0.486036,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,1106,0.487992,1.000000,0.570180,0.439121,1.0,1.000000,0.535368,1.000000,1.0,0.266178,1.000000,0.119048,1.000000,1.0,1.000000,0.615929,0.777778,0.500,0.884058,0.750000,1.000000,1.0,0.410628,0.374684,0.526866,0.262700,0.526536,1.000000,1.000000,0.139959,1.0,0.198381,0.946372,0.182849,1.0,0.000000,0.184503,0.239444,1.0,1.000000,1.0,1.000000,0.568437,0.543341,0.0,0.728921,0.333333,0.0,0.666667,0.5,0.375,0.333333,0.786355,0.583333,1.0,0.666667,0.444444,1.000000,0.854545,0.661058,0.50,0.502116,1.000000,1.000000,1.0,0.217036,0.058501,0.000000,0.0,0.000000,0.0,1.0,1.000000,1.000000,0.000000,0.272727,1.00,1.0,1.000000,0.728982,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,414,0.180103,0.182609,0.363044,0.377814,1.0,0.013158,1.000000,1.000000,1.0,1.000000,1.000000,0.464286,0.025278,1.0,1.000000,1.000000,0.444444,0.625,0.398551,0.000000,1.000000,1.0,0.031401,0.048101,1.000000,0.000000,1.000000,1.000000,0.968750,1.000000,1.0,1.000000,1.000000,0.000000,1.0,0.000000,0.431507,0.164975,1.0,0.333333,1.0,0.060918,0.425446,0.000000,0.0,0.397696,0.000000,0.0,0.333333,0.0,0.250,0.333333,1.000000,0.250000,1.0,0.333333,0.544256,0.433824,0.245455,1.000000,0.50,0.253879,1.000000,1.000000,1.0,0.000000,0.000000,0.235507,0.0,0.000000,0.0,1.0,1.000000,1.000000,0.000000,0.181818,1.00,1.0,1.000000,0.389574,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
3,523,0.407007,0.182609,0.321097,0.263645,1.0,1.000000,1.000000,1.000000,1.0,0.266178,1.000000,0.190476,0.051567,1.0,1.000000,0.192920,0.555556,0.750,0.543478,0.000000,1.000000,1.0,0.094203,0.073418,1.000000,0.000000,1.000000,0.116371,0.968750,1.000000,1.0,1.000000,0.473186,0.070695,1.0,0.000000,0.258990,0.164321,1.0,1.000000,1.0,1.000000,0.416506,0.319613,0.0,0.568066,0.000000,0.0,0.666667,0.0,0.375,0.333333,1.000000,0.416667,1.0,0.666667,0.544256,0.433824,0.454545,1.000000,0.50,0.296192,1.000000,1.000000,1.0,0.000000,0.043876,0.065217,0.0,0.000000,0.0,1.0,1.000000,1.000000,0.000000,0.818182,0.00,1.0,1.000000,0.495416,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,1037,0.000000,1.000000,0.534524,0.449114,1.0,1.000

## Output files

Each run creates an `outputs` folder automatically if it does not already exist.

The final files are timestamped so previous runs are not overwritten:

- `X_train_YYYYMMDD_HHMMSS.csv`
- `X_test_YYYYMMDD_HHMMSS.csv`

The unsupervised preprocessing parameters are fitted only on the training split
and reused for the test split.

No target variable, target encoding, one-hot expansion, or Lasso feature selection
is used.
